[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.fr/cap06/cap06.EPs_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

## 💻 Partie pratique avec exercices de programmation

Les exercices de programmation (EP) de cette section complètent les concepts présentés tout au long du chapitre 6 à travers la mise en œuvre d'algorithmes liés à l'inspection industrielle et à l'analyse de documents. L'objectif est de consolider les fondamentaux étudiés, en reproduisant, à échelle réduite, les étapes d'un *pipeline* typique de Vision par Ordinateur.

Contrairement aux chapitres précédents, dont les exercices mettaient l'accent sur des opérations plus directement liées aux données d'image, les EP de ce chapitre se concentrent sur les **grandeurs intermédiaires** produites au cours du traitement, telles que les aires, les périmètres, la circularité, les angles de droites, les degrés de remplissage de bulles, les cartes de variance et les cartes de différence. Cette approche permet de comprendre et de valider chaque étape du *pipeline* de manière indépendante, sans dépendre de bibliothèques spécialisées pour l'acquisition d'images, la détection de marqueurs ou le décodage de codes — à l'exception de l'exercice de clôture du chapitre (EP06_08), qui introduit délibérément l'utilisation d'OpenCV pour la segmentation et le décodage réel d'un *QRCode*, bouclant ainsi la boucle entre les concepts théoriques et les outils employés en pratique.

Les exercices suivent la même séquence conceptuelle que le chapitre, par ordre croissant de complexité. Dans un premier temps, sont abordées les métriques d'évaluation de segmentation, utilisées pour quantifier la qualité des masques binaires. Ensuite, sont étudiés les critères géométriques pour la sélection de marqueurs, la classification de marquages dans des formulaires et l'estimation de l'inclinaison de documents au moyen de la Transformée de Hough. Dans la partie finale, les exercices explorent la normalisation de l'éclairage, la détection de défauts par analyse de texture et l'intégration entre le recalage géométrique et la soustraction d'images dans un *pipeline* simplifié d'inspection industrielle.

Chaque exercice représente une étape isolée d'un système réel de Vision par Ordinateur, permettant de valider individuellement des concepts qui, dans les applications industrielles, sont combinés en un seul *pipeline* d'inspection.

### 🗺️ Légende de difficulté

| Niveau | Signification | EP |
|:---:|---|---|
| 🟢 | Très facile / facile — mise en œuvre d'un seul concept ou d'un algorithme simple | EP06_01, EP06_02 |
| 🟡 | Facile–moyen — traitement de multiples cas ou utilisation de critères statistiques simples | EP06_03, EP06_04 |
| 🟠 | Moyen — traitement matriciel point à point | EP06_05 |
| 🔴 | Difficile — traitement matriciel avec opérations de voisinage (fenêtre glissante) | EP06_06 |
| 🟣 | Très difficile — intégration de multiples étapes d'un *pipeline* de Vision par Ordinateur | EP06_07 |
| ⚫ | Spécial — utilisation d'une bibliothèque spécialisée (`cv2`) pour la segmentation géométrique et le décodage réel de code-barres/QRCode | EP06_08 |

> ### ❗ Directives pour la résolution des exercices de programmation
>
> Sauf indication contraire, tous les exercices utilisent la convention de coordonnées matricielles `[ligne][colonne]`, avec l'origine en $(0,0)$ dans le coin supérieur gauche de l'image.
>
> Lorsqu'un arrondi numérique est nécessaire, on doit utiliser l'arrondi standard à l'entier le plus proche (*round half away from zero*, avec `np.floor(img + 0.5)`). Les comparaisons avec des seuils (par exemple, circularité, variance, différence d'intensité ou degré de remplissage) doivent être considérées comme **strictes** (`>`), sauf si l'énoncé spécifie explicitement un autre critère.
>
> Chaque exercice a été conçu pour mettre l'accent sur un concept spécifique présenté dans le chapitre. Il est recommandé de mettre en œuvre initialement la solution de manière directe et, seulement après sa validation, de rechercher des alternatives plus efficaces ou plus générales.

### 🎯 Objectif de ce cahier

Ce cahier a été élaboré pour accompagner le développement, la validation et les tests des solutions des **Exercices de Programmation (EPs)** dans un environnement interactif, tel que Google Colab ou Jupyter Notebook. Après avoir vérifié le fonctionnement de l’implémentation avec les cas de test présentés, le code peut être soumis à Moodle pour l’évaluation officielle.

#### *Download*

Exécutez la cellule suivante pour obtenir les fichiers `morph.py` et `testsuite.py`, utilisés par les exercices de ce chapitre.

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Exécution des Tests

Après avoir implémenté la solution, exécutez `TestSuite("EP06_01.extensão").run()` dans une nouvelle cellule, en remplaçant `extensão` par le langage utilisé (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). Le système récupère automatiquement les cas de test du dépôt du cours, exécute le programme et affiche le résultat de l'évaluation.

En Python, il est également possible de tester la solution directement à partir d'une *chaîne de caractères*, sans avoir besoin d'enregistrer le code dans un fichier. Pour cela, stockez le programme dans une variable et utilisez la méthode `run_code` :

```python
codigo = """
# ... votre code ici ...
"""

TestSuite("EP06_01").run_code(codigo)
```

### EP06_01 🟢 Évaluation de Segmentation par IoU (*Intersection over Union*)

Tout au long de ce chapitre, plusieurs étapes du *pipeline* produisent des **masques binaires**, comme dans la segmentation de documents, la localisation de *QRCodes* et la détection de défauts. Pour évaluer objectivement la qualité de ces segmentations, il est nécessaire de les comparer à un masque de référence (*ground truth*).

L'une des métriques les plus utilisées à cette fin est l'**IoU** (*Intersection over Union*, ou Intersection sur Union), définie comme le rapport entre l'aire de l'intersection et l'aire de l'union de deux masques binaires. Plus la valeur de l'IoU est élevée, plus la concordance entre la segmentation produite par l'algorithme et la référence est grande.

#### 📋 Directives d'Implémentation

1. **Dimensions :** Lire les entiers $L$ (nombre de lignes) et $C$ (nombre de colonnes).
2. **Masque de référence :** Lire les $L \times C$ éléments binaires (0 ou 1) de la matrice `ref`.
3. **Masque prédit :** Lire les $L \times C$ éléments binaires (0 ou 1) de la matrice `pred`.
4. **Intersection :** Compter le nombre de positions $(i,j)$ pour lesquelles `ref[i][j] = 1` et `pred[i][j] = 1`.
5. **Union :** Compter le nombre de positions $(i,j)$ pour lesquelles `ref[i][j] = 1` ou `pred[i][j] = 1`.
6. **Cas dégénéré :** Si l'union est égale à $0$, définir $\mathrm{IoU}=1{,}0$, car les deux masques sont vides.
7. **Calcul :** Si l'union est supérieure à zéro, calculer

$$
\mathrm{IoU}=
\frac{|\mathrm{Intersecao}|}
{|\mathrm{Uniao}|}.
$$

8. **Classification :** Déterminer la classification qualitative en utilisant la valeur de l'IoU **avant** l'arrondi.
9. **Arrondi :** Afficher l'IoU avec quatre décimales.
10. **Sortie :** Imprimer, dans cet ordre, l'intersection, l'union, l'IoU et la classification.

#### 📌 Contraintes Computationnelles

- Si l'union est égale à $0$, la division ne doit pas être effectuée ; l'IoU doit être définie comme $1{,}0$.
- Les plages de classification utilisent des comparaisons non strictes ($\geq$).
- La classification doit être effectuée en utilisant la valeur de l'IoU en précision complète, avant l'arrondi pour l'affichage.

#### 🧠 Fondements Théoriques

L'IoU est définie par

$$
\mathrm{IoU}=
\frac{|R\cap P|}
{|R\cup P|},
$$

où :

- $R$ représente l'ensemble des pixels appartenant au masque de référence ;
- $P$ représente l'ensemble des pixels appartenant au masque prédit ;
- $|R\cap P|$ correspond au nombre de pixels appartenant simultanément aux deux masques ;
- $|R\cup P|$ correspond au nombre de pixels appartenant à au moins l'un des masques.

| Plage d'IoU | Classification | Interprétation |
|---|---|---|
| $\mathrm{IoU}\geq0{,}90$ | `EXCELENTE` | Concordance très élevée entre les masques. |
| $0{,}70\leq\mathrm{IoU}<0{,}90$ | `BOM` | Petites différences entre les masques. |
| $0{,}50\leq\mathrm{IoU}<0{,}70$ | `ACEITAVEL` | Concordance partielle entre les masques. |
| $\mathrm{IoU}<0{,}50$ | `RUIM` | Faible concordance entre les masques. |

L'IoU dépend uniquement du chevauchement entre les masques et est donc indépendante de la taille de l'image.

#### 📦 Spécification d'Entrée et de Sortie (VPL)

**Entrée :**

- Ligne 1 : entier $L$.
- Ligne 2 : entier $C$.
- $L$ lignes suivantes : éléments binaires (0 ou 1) de la matrice `ref`.
- $L$ lignes suivantes : éléments binaires (0 ou 1) de la matrice `pred`.

**Sortie :**

- Ligne 1 : `Intersecao: X`
- Ligne 2 : `Uniao: Y`
- Ligne 3 : `IoU: Z`
- Ligne 4 : `Classificacao: NOME`

La valeur de `IoU` doit être imprimée avec quatre décimales.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 2<br>2<br>1 1<br>0 0<br>1 0<br>0 0 | Intersecao: 1<br>Uniao: 2<br>IoU: 0.5000<br>Classificacao: ACEITAVEL | La moitié de la région de référence a été correctement segmentée. |
| 2<br>2<br>0 0<br>0 0<br>0 0<br>0 0 | Intersecao: 0<br>Uniao: 0<br>IoU: 1.0000<br>Classificacao: EXCELENTE | Les deux masques sont vides ; par convention, $\mathrm{IoU}=1{,}0$. |

In [ ]:
from IPython.display import HTML

HTML("""
<div id="sim-ep0601" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0601 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0601 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0601 button:hover { background: #e8dfcf; }
  #sim-ep0601 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0601_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0601_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0601_grid_ctrls { display: grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr)); gap: 12px; }
  .sim-ep0601_px { width: 24px; height: 24px; border: 1px solid #e4dcc8; box-sizing: border-border; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP06_01 : IoU (Intersection over Union)</span>
  <span class="sim-ep0601_pill">IoU = |A &cap; B| / |A &cup; B|</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Controles -->
  <div class="sim-ep0601_panel" style="margin-bottom:14px;">
    <div class="sim-ep0601_grid_ctrls">
      
      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Déplacement H (&Delta;x)</label>
          <span id="sim-ep0601_vdx" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0601_dx" type="range" min="-3" max="3" step="1" value="0">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Déplacement V (&Delta;y)</label>
          <span id="sim-ep0601_vdy" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0601_dy" type="range" min="-3" max="3" step="1" value="0">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Côté du carré</label>
          <span id="sim-ep0601_vsz" style="font-family:monospace; font-weight:700; color:#26241d;">6</span>
        </div>
        <input id="sim-ep0601_sz" type="range" min="2" max="8" step="1" value="6">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:10px; text-align:center;">
      Déplacez et redimensionnez le masque prédit pour évaluer l'alignement.
    </div>
  </div>

  <!-- Exibição das Máscaras 10x10 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Référence (A)
      </div>
      <div id="sim-ep0601_ref" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Prédit (B)
      </div>
      <div id="sim-ep0601_pred" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Chevauchement (A &cap; B)
      </div>
      <div id="sim-ep0601_mix" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0601_dbg" class="sim-ep0601_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep01(root){
    if (!root || root.dataset.sim06Ep01Init) return;
    root.dataset.sim06Ep01Init = "1";

    var dx = root.querySelector("#sim-ep0601_dx");
    var dy = root.querySelector("#sim-ep0601_dy");
    var sz = root.querySelector("#sim-ep0601_sz");

    var vdx = root.querySelector("#sim-ep0601_vdx");
    var vdy = root.querySelector("#sim-ep0601_vdy");
    var vsz = root.querySelector("#sim-ep0601_vsz");

    var gRef  = root.querySelector("#sim-ep0601_ref");
    var gPred = root.querySelector("#sim-ep0601_pred");
    var gMix  = root.querySelector("#sim-ep0601_mix");

    var dbg = root.querySelector("#sim-ep0601_dbg");

    var N = 10;
    var ref = {x: 2, y: 2, w: 6, h: 6};

    function inside(x, y, r){
      return x >= r.x && x < r.x + r.w && y >= r.y && y < r.y + r.h;
    }

    function pixel(color){
      var d = document.createElement("div");
      d.className = "sim-ep0601_px";
      d.style.background = color;
      return d;
    }

    function classe(i){
      if (i >= 0.90) return "Excelente";
      if (i >= 0.75) return "Muito boa";
      if (i >= 0.50) return "Aceitável";
      return "Ruim";
    }

    function render(){
      vdx.textContent = dx.value;
      vdy.textContent = dy.value;
      vsz.textContent = sz.value;

      gRef.innerHTML  = "";
      gPred.innerHTML = "";
      gMix.innerHTML  = "";

      var pred = {
        x: ref.x + parseInt(dx.value, 10),
        y: ref.y + parseInt(dy.value, 10),
        w: parseInt(sz.value, 10),
        h: parseInt(sz.value, 10)
      };

      var inter = 0;
      var uniao = 0;

      for (var y = 0; y < N; y++){
        for (var x = 0; x < N; x++){
          var r = inside(x, y, ref);
          var p = inside(x, y, pred);

          gRef.appendChild(pixel(r ? "#7fdc92" : "#ffffff"));
          gPred.appendChild(pixel(p ? "#7fbfff" : "#ffffff"));

          if (r && p){
            gMix.appendChild(pixel("#9b59b6"));
            inter++;
          }
          else if (r){
            gMix.appendChild(pixel("#7fdc92"));
            uniao++;
          }
          else if (p){
            gMix.appendChild(pixel("#7fbfff"));
            uniao++;
          }
          else{
            gMix.appendChild(pixel("#ffffff"));
          }

          if (r && p) uniao++;
        }
      }

      var iou = inter / uniao;

      dbg.innerHTML =
        "<b>Interseção</b> = " + inter + " pixels &nbsp;&nbsp;&nbsp;" +
        "<b>União</b> = " + uniao + " pixels<br><br>" +
        "IoU = <b>" + inter + " / " + uniao + " = " + iou.toFixed(4) + "</b><br><br>" +
        "<span style='font-weight:700; color:#04342C;'>" + classe(iou) + "</span>";
    }

    dx.addEventListener('input', render);
    dy.addEventListener('input', render);
    sz.addEventListener('input', render);

    render();
  }

  function tryInitSim06Ep01(){
    var root = document.getElementById('sim-ep0601');
    if (root) initSim06Ep01(root); else setTimeout(tryInitSim06Ep01, 200);
  }
  tryInitSim06Ep01();
})();
</script>
""")

**Figure 6.1:** Simulateur EP06_01 : IoU entre masque de référence et masque prédit


<figure id="fig-06-sim-ep0601">
  <img src="imagens/fig-06-sim-ep0601.png" alt=" Simulateur EP06_01 : IoU entre masque de référence et masque prédit " style="max-width:80%" />
  <figcaption><strong>Figure 6.1:</strong>  Simulateur EP06_01 : IoU entre masque de référence et masque prédit </figcaption>
</figure>

In [ ]:
%%writefile EP06_01.py
# Code Python

In [ ]:
TestSuite("EP06_01.py").run()

### EP06_02 🟢 Filtrage des marqueurs par circularité

Après la segmentation d'une image, il est courant que plusieurs composants connexes soient identifiés. Dans des applications telles que la rectification de documents, seuls certains de ces composants correspondent aux marqueurs de référence utilisés pour l'alignement de l'image. Un critère fréquemment employé pour sélectionner ces marqueurs est la **circularité**, qui mesure à quel point la forme d'un composant est proche d'un cercle.

Dans cet exercice, chaque composant est décrit par son aire $A$ et son périmètre $P$. L'objectif est de calculer sa circularité et de décider, à partir d'un seuil fourni, si le composant doit être accepté ou rejeté comme candidat marqueur.

#### 📋 Directives d'implémentation

1. **Quantité :** Lire l'entier $N$ (nombre de candidats) et le seuil de circularité $C_{\text{limiar}}$ (nombre réel).
2. **Données des candidats :** Pour chacun des $N$ candidats, lire l'aire $A$ (entier) et le périmètre $P$ (nombre réel).
3. **Circularité :** Calculer $C=\frac{4\pi A}{P^2}$, où :

- $A$ est l'aire du composant ;
- $P$ est le périmètre du composant ;
- $C$ est la circularité.

4. **Cas dégénéré :** Si $P=0$, considérer $C=0$ et classer directement le candidat comme `REJETÉ`.
5. **Classification :** Si $C>C_{\text{limiar}}$, classer le candidat comme `ACCEPTÉ` ; sinon, le classer comme `REJETÉ`.
6. **Arrondi :** Afficher la valeur de $C$ avec quatre décimales.
7. **Sortie :** Pour chaque candidat, imprimer la valeur de $C$ suivie de la classification. À la fin, imprimer le nombre total de candidats acceptés.

#### 📌 Contraintes computationnelles

- Utiliser la constante $\pi$ de la bibliothèque standard du langage (par exemple, `math.pi`), sans approximations.
- La comparaison doit être effectuée avec la valeur de $C$ en précision complète, avant l'arrondi pour l'affichage.
- Le critère d'acceptation est strict ($C>C_{\text{limiar}}$).
- Si $P=0$, la division ne doit pas être effectuée.

#### 🧠 Fondement théorique

La circularité est un descripteur géométrique défini par $C=\frac{4\pi A}{P^2}$, où :

- $A$ est l'aire du composant ;
- $P$ est le périmètre du composant ;
- $C$ est la circularité.

Pour un cercle parfait, $C=1$. À mesure que la forme devient plus allongée ou irrégulière, le périmètre croît plus rapidement que l'aire, réduisant la valeur de $C$.

| Forme | Circularité approximative | Interprétation |
|---|---:|---|
| Cercle | $1{,}0000$ | Forme circulaire. |
| Carré | $0{,}7854$ | Forme approximativement compacte. |
| Forme allongée ou irrégulière | $C\ll1$ | Faible circularité. |
| $P=0$ | $0$ (convention adoptée) | Contour dégénéré. |

La circularité est invariante par translation, rotation et mise à l'échelle, et est largement utilisée pour distinguer les composants approximativement circulaires des autres formats.

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

- Ligne 1 : entier $N$.
- Ligne 2 : nombre réel $C_{\text{limiar}}$.
- Les $N$ lignes suivantes : aire $A$ (entier) et périmètre $P$ (réel), séparés par un espace.

**Sortie :**

- Une ligne pour chaque candidat, au format `C ACCEPTÉ` ou `C REJETÉ`, avec $C$ présenté avec quatre décimales.
- Dernière ligne : `Total acceptés : X`.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 3<br>0.6<br>78 31.4<br>100 40<br>50 60 | 0.9941 ACCEPTÉ<br>0.7854 ACCEPTÉ<br>0.1745 REJETÉ<br>Total acceptés : 2 | Candidat approximativement circulaire, forme compacte et forme allongée. |
| 1<br>0.9<br>10 0 | 0.0000 REJETÉ<br>Total acceptés : 0 | Périmètre nul : contour dégénéré. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0602" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0602 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0602 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0602 button:hover { background: #e8dfcf; }
  #sim-ep0602 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0602_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0602_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP06_02 : Filtre de marqueurs par circularité</span>
  <span class="sim-ep0602_pill">C = 4&pi;A / P&sup2;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0602_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Seuil de circularité (C_seuil) : <span id="sim-ep0602_vl" style="font-family:monospace; color:#26241d;">0.60</span>
      </label>
    </div>
    
    <input id="sim-ep0602_sl" type="range" min="0.05" max="0.99" step="0.01" value="0.60">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajustez le seuil et observez quels candidats (disques, carrés et formes irrégulières) survivent au filtre.
    </div>
  </div>

  <!-- Cards de Candidatos -->
  <div id="sim-ep0602_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(100px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0602_debug" class="sim-ep0602_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep02(root){
    if (!root || root.dataset.sim06Ep02Init) return;
    root.dataset.sim06Ep02Init = "1";

    var candidatos = [
      {nome: "Disco", A: 78, P: 31.4},
      {nome: "Quadrado", A: 100, P: 40},
      {nome: "Retângulo", A: 60, P: 44},
      {nome: "Rasura", A: 50, P: 60},
      {nome: "Ponto", A: 10, P: 0}
    ];

    var slEl  = root.querySelector('#sim-ep0602_sl');
    var vlEl  = root.querySelector('#sim-ep0602_vl');
    var cards = root.querySelector('#sim-ep0602_cards');
    var dbg   = root.querySelector('#sim-ep0602_debug');

    function render(){
      var th = parseFloat(slEl.value);
      vlEl.textContent = th.toFixed(2);
      cards.innerHTML = '';
      var aceitos = 0;

      candidatos.forEach(function(c){
        var C = (c.P === 0) ? 0 : (4 * Math.PI * c.A) / (c.P * c.P);
        var ok = c.P !== 0 && C > th;
        if (ok) aceitos++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (ok ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');
        
        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + c.nome + '</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px; opacity:0.8;">A = ' + c.A + '<br>P = ' + c.P + '</div>' +
          '<div style="font-family:monospace; font-weight:700; margin-bottom:4px;">C = ' + C.toFixed(4) + '</div>' +
          '<div style="font-weight:700; font-size:10px; letter-spacing:0.04em;">' + (ok ? 'ACEITO' : 'REJEITADO') + '</div>';
        
        cards.appendChild(div);
      });

      dbg.textContent = 'C_seuil = ' + th.toFixed(2) + '  |  Candidatos aceitos: ' + aceitos + ' / ' + candidatos.length;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep02(){
    var root = document.getElementById('sim-ep0602');
    if (root) initSim06Ep02(root); else setTimeout(tryInitSim06Ep02, 200);
  }
  tryInitSim06Ep02();
})();
</script>
""")

**Figure 6.2:** Simulateur EP06_02 : Filtre de Marqueurs par Circularité


<figure id="fig-06-sim-ep0602">
  <img src="imagens/fig-06-sim-ep0602.png" alt=" Simulateur EP06_02 : Filtre de Marqueurs par Circularité " style="max-width:80%" />
  <figcaption><strong>Figure 6.2:</strong>  Simulateur EP06_02 : Filtre de Marqueurs par Circularité </figcaption>
</figure>

In [ ]:
%%writefile EP06_02.py
# Code Python

In [ ]:
TestSuite("EP06_02.py").run()

### EP06_03 🟡 Classification des Marques sur les Feuilles de Réponses (OMR)

Après la correction de la feuille et la segmentation des cadres de réponses, le MCTest estime, pour chaque bulle, un **degré de remplissage**, représenté par une valeur entre $0$ et $100$. À partir de ces valeurs, le système doit déterminer automatiquement l’alternative marquée, en identifiant également les questions blanches et les cas de marques multiples.

Dans cet exercice, vous implémenterez cette étape de décision du *pipeline* OMR. La classification dépend d’un seuil de remplissage : de petites variations de cette valeur peuvent modifier le résultat de la lecture automatique.

#### 📋 Directives d’Implémentation

1. **Paramètres :** Lire les entiers $Q$ (nombre de questions) et $K$ (nombre d’alternatives par question, avec $2 \le K \le 26$) ainsi que le seuil de remplissage $\mathrm{Th}$ (nombre réel entre $0$ et $100$).
2. **Degrés de remplissage :** Pour chacune des $Q$ questions, lire les $K$ valeurs réelles correspondant aux alternatives `A`, `B`, `C`, ..., dans l’ordre de saisie.
3. **Comptage des marques :** Pour chaque question, compter combien d’alternatives ont un degré de remplissage **strictement supérieur** à $\mathrm{Th}$.
4. **Classification :**
   - Si aucune alternative ne dépasse $\mathrm{Th}$, classer la question comme `BRANCO`.
   - Si exactement une alternative dépasse $\mathrm{Th}$, imprimer la lettre correspondante (`A`, `B`, `C`, ...).
   - Si deux alternatives ou plus dépassent $\mathrm{Th}$, classer la question comme `DUPLA_MARCACAO`.
5. **Sortie par question :** Imprimer, dans l’ordre de lecture, la classification de chaque question.
6. **Totaux :** À la fin, imprimer le nombre de questions `OK` (une seule marque), `BRANCO` et `DUPLA_MARCACAO`.

#### 📌 Contraintes Computationnelles

* **Comparaison stricte :** seules les valeurs supérieures à $\mathrm{Th}$ sont considérées comme des marques valides ; les valeurs exactement égales au seuil ne doivent pas être comptabilisées.
* **Lettres des alternatives :** l’indice $0$ correspond à l’alternative `A`, l’indice $1$ à l’alternative `B`, et ainsi de suite.
* **Marques multiples :** dès que deux alternatives ou plus dépassent le seuil, la classification doit être `DUPLA_MARCACAO`, indépendamment de leurs degrés de remplissage respectifs.

#### 🧠 Fondement Théorique

| Situation | Classification | Interprétation |
|---|---|---|
| Exactement une alternative au-dessus du seuil | Lettre de l’alternative | Réponse valide |
| Aucune alternative au-dessus du seuil | `BRANCO` | Question non répondue |
| Deux alternatives ou plus au-dessus du seuil | `DUPLA_MARCACAO` | Réponse ambiguë |

Le seuil de remplissage contrôle la sensibilité de l’algorithme. Des valeurs très basses tendent à augmenter le nombre de `DUPLA_MARCACAO`, tandis que des valeurs très hautes peuvent augmenter la quantité de questions classées comme `BRANCO`.

#### 📦 Spécification d’Entrée et de Sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $Q$.
* Ligne 2 : Entier $K$.
* Ligne 3 : Nombre réel $\mathrm{Th}$.
* Les $Q$ lignes suivantes : $K$ nombres réels, correspondant aux degrés de remplissage des alternatives.
* Ligne 1 : Entiers $Q$ et $K$.

**Sortie :**

* $Q$ lignes, chacune contenant la classification de la question respective.
* Ligne finale : `OK: x  BRANCO: y  DUPLA_MARCACAO: z`.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 3<br>4<br>50<br>10 85 5 12<br>20 15 18 22<br>90 88 10 5 | B<br>BRANCO<br>DUPLA_MARCACAO<br>OK: 1  BRANCO: 1  DUPLA_MARCACAO: 1 | Dans la première question, seule `B` dépasse le seuil ; dans la deuxième, aucune alternative ne le dépasse ; dans la troisième, `A` et `B` dépassent le seuil. |
| 1<br>2<br>50.0<br>50 50 | BRANCO<br>OK: 0  BRANCO: 1  DUPLA_MARCACAO: 0 | Les valeurs égales au seuil ne sont pas considérées comme des marques valides. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0603" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0603 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0603 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0603 button:hover { background: #e8dfcf; }
  #sim-ep0603 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0603_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0603_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP06_03 : Classification des marques OMR</span>
  <span class="sim-ep0603_pill">4 alternatives</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0603_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Seuil de remplissage (Th) : <span id="sim-ep0603_vth" style="font-family:monospace; color:#26241d;">50</span>%
      </label>
    </div>
    
    <input id="sim-ep0603_th" type="range" min="0" max="100" step="1" value="50">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajustez le degré de remplissage de chaque bulle (A&ndash;D) et le seuil pour observer la classification résultante.
    </div>
  </div>

  <!-- Sliders das Bolhas (A-D) -->
  <div class="sim-ep0603_panel" style="margin-bottom:14px;">
    <div id="sim-ep0603_bubbles" style="display:grid; grid-template-columns:repeat(4, 1fr); gap:12px;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0603_debug" class="sim-ep0603_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep03(root){
    if (!root || root.dataset.sim06Ep03Init) return;
    root.dataset.sim06Ep03Init = "1";

    var letras = ['A', 'B', 'C', 'D'];
    var valores = [10, 85, 5, 12];
    var thEl  = root.querySelector('#sim-ep0603_th');
    var vthEl = root.querySelector('#sim-ep0603_vth');
    var box   = root.querySelector('#sim-ep0603_bubbles');
    var dbg   = root.querySelector('#sim-ep0603_debug');

    box.innerHTML = '';
    var sliders = [];

    letras.forEach(function(L, i){
      var col = document.createElement('div');
      col.style.cssText = 'text-align:center; background:#fafaf7; border:1px solid #e9e3d3; padding:10px; border-radius:8px;';
      col.innerHTML = '<div style="font-weight:700; font-size:12px; color:#5e5a4a; margin-bottom:6px;">' + L + '</div>' +
        '<input type="range" min="0" max="100" step="1" value="' + valores[i] + '" id="sim-ep0603_b' + i + '">' +
        '<div id="sim-ep0603_v' + i + '" style="font-family:monospace; font-weight:700; font-size:11px; color:#26241d; margin-top:6px;">' + valores[i] + '%</div>';
      box.appendChild(col);
      sliders.push(col.querySelector('#sim-ep0603_b' + i));
    });

    function render(){
      var th = parseFloat(thEl.value);
      vthEl.textContent = th.toFixed(0);
      var marcadas = [];

      sliders.forEach(function(s, i){
        var v = parseFloat(s.value);
        root.querySelector('#sim-ep0603_v' + i).textContent = v.toFixed(0) + '%';
        if (v > th) marcadas.push(letras[i]);
      });

      var resultado;
      if (marcadas.length === 0) {
        resultado = 'BRANCO';
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#8a8371';
      } else if (marcadas.length === 1) {
        resultado = 'RESPOSTA: ' + marcadas[0];
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        resultado = 'DUPLA_MARCACAO (' + marcadas.join(', ') + ')';
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'Classification de la question : ' + resultado;
    }

    sliders.forEach(function(s){ s.addEventListener('input', render); });
    thEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep03(){
    var root = document.getElementById('sim-ep0603');
    if (root) initSim06Ep03(root); else setTimeout(tryInitSim06Ep03, 200);
  }
  tryInitSim06Ep03();
})();
</script>
""")

**Figure 6.3:** Simulateur EP06_03 : Classification des marquages OMR


<figure id="fig-06-sim-ep0603">
  <img src="imagens/fig-06-sim-ep0603.png" alt=" Simulateur EP06_03 : Classification des marquages OMR " style="max-width:80%" />
  <figcaption><strong>Figure 6.3:</strong>  Simulateur EP06_03 : Classification des marquages OMR </figcaption>
</figure>

In [ ]:
%%writefile EP06_03.py
# Code Python

In [ ]:
TestSuite("EP06_03.py").run()

### EP06_04 🟡 Estimateur d'Inclinaison par Médiane Angulaire (*Deskew*)

Après la détection des contours et l'application de la Transformée de Hough, on obtient un ensemble de droites candidates pour l'orientation prédominante du document. Chaque droite fournit une estimation de l'angle d'inclinaison, calculée par

$$
\text{angle} = \operatorname{rad2deg}(\theta) - 90.
$$

Cependant, toutes les droites ne correspondent pas aux lignes du document : certaines résultent de bruits, d'ombres ou d'autres éléments de l'image. Dans cet exercice, vous implémenterez l'étape d'estimation robuste de l'angle d'inclinaison, en filtrant les valeurs plausibles et en calculant leur médiane.

#### 📋 Directives d'Implémentation

1. **Quantité :** Lire l'entier $M$, correspondant au nombre d'angles estimés.
2. **Angles :** Lire les $M$ valeurs réelles, en degrés.
3. **Filtrage :** Conserver uniquement les angles qui satisfont **strictement** $-45 < \text{angle} < 45$.
4. **Absence de candidats :** Si aucun angle ne subsiste après le filtrage, imprimer exactement `SEM_CORRECAO`.
5. **Médiane :** S'il existe des angles valides :
   - si la quantité est impaire, la médiane est l'élément central de la séquence ordonnée ;
   - si elle est paire, la médiane est la moyenne arithmétique des deux éléments centraux.
6. **Sortie :** Imprimer la médiane arrondie à deux décimales (arrondi standard, *round half away from zero*, avec `np.floor(img + 0.5)`).

#### 📌 Contraintes Computationnelles

* **Intervalle ouvert :** les angles égaux à $-45$ ou $45$ ne doivent pas être considérés.
* **Précision :** calculer la médiane en utilisant les valeurs originales ; l'arrondi doit être effectué uniquement en sortie.
* **Cas vide :** s'il n'y a pas d'angles valides, aucune médiane ne doit être calculée.

#### 🧠 Fondement Théorique

| Situation | Résultat |
|---|---|
| Majorité des angles concentrée autour de l'inclinaison réelle | La médiane approxime l'orientation du document. |
| Peu d'angles aberrants (*outliers*) | La médiane subit peu d'influence de ces valeurs. |
| Angles hors de l'intervalle $(-45^\circ,45^\circ)$ | Ils sont écartés avant le calcul. |
| Aucun angle valide | Aucune correction n'est appliquée (`SEM_CORRECAO`). |

La médiane est utilisée car elle est plus robuste que la moyenne en présence de quelques valeurs aberrantes, produisant une estimation plus stable de l'inclinaison prédominante du document.

#### 📦 Spécification d'Entrée et de Sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $M$.
* Ligne 2 : $M$ nombres réels, correspondant aux angles en degrés.

**Sortie :**

* Une seule ligne contenant l'angle estimé, avec deux décimales, ou le mot `SEM_CORRECAO` si aucun angle n'est valide.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 5<br>-50 -10.5 2.3 2.3 47 | 2.30 | Seuls les angles dans l'intervalle $(-45,45)$ sont considérés ; la médiane est $2{,}3$. |
| 4<br>-46 50 45 -45 | SEM_CORRECAO | Aucun angle n'appartient à l'intervalle ouvert $(-45,45)$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0604" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0604 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0604 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0604 button:hover { background: #e8dfcf; }
  #sim-ep0604 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0604_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0604_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP06_04 : Estimateur d'Inclinaison par Médiane Angulaire (Deskew)</span>
  <span class="sim-ep0604_pill">médiane(-45&deg; &lt; &theta; &lt; 45&deg;)</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0604_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Angle du Bruit Supplémentaire (&theta;_bruit) : <span id="sim-ep0604_vl" style="font-family:monospace; color:#26241d;">47</span>&deg;
      </label>
    </div>
    
    <input id="sim-ep0604_sl" type="range" min="-80" max="80" step="1" value="47">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Faites glisser l'angle du bruit supplémentaire à l'intérieur ou à l'extérieur de l'intervalle [-45&deg;, +45&deg;] et observez comment la médiane reste stable.
    </div>
  </div>

  <!-- Exibição dos Ângulos Amostrados -->
  <div class="sim-ep0604_panel" style="margin-bottom:14px;">
    <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; text-align:center; letter-spacing:0.04em;">
      Échantillons d'Angles (Vert = Dans la Plage, Rouge = Bruit Écarté)
    </div>
    <div id="sim-ep0604_pts" style="display:flex; gap:8px; flex-wrap:wrap; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0604_debug" class="sim-ep0604_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep04(root){
    if (!root || root.dataset.sim06Ep04Init) return;
    root.dataset.sim06Ep04Init = "1";

    var base = [-10.5, 2.3, 2.3];
    var slEl  = root.querySelector('#sim-ep0604_sl');
    var vlEl  = root.querySelector('#sim-ep0604_vl');
    var ptsEl = root.querySelector('#sim-ep0604_pts');
    var dbg   = root.querySelector('#sim-ep0604_debug');

    function median(arr){
      var a = arr.slice().sort(function(x, y){ return x - y; });
      var n = a.length;
      if (n === 0) return null;
      var mid = Math.floor(n / 2);
      return (n % 2 === 1) ? a[mid] : (a[mid - 1] + a[mid]) / 2;
    }

    function render(){
      var extra = parseFloat(slEl.value);
      vlEl.textContent = extra;
      var todos = base.concat([extra, -50]);
      var validos = todos.filter(function(a){ return a > -45 && a < 45; });
      
      ptsEl.innerHTML = '';
      todos.forEach(function(a){
        var ok = a > -45 && a < 45;
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 12px; border-radius:8px; font-family:monospace; font-size:12px; font-weight:700; transition:all 0.15s ease;' +
          (ok ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');
        div.textContent = a + '°';
        ptsEl.appendChild(div);
      });

      var med = median(validos);

      if (med === null) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
        dbg.textContent = 'Válidos: []  |  Mediana estimada: SEM_CORRECAO';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
        dbg.textContent = 'Valides : [' + validos.join(', ') + ']  |  Mediana estimada: ' + med.toFixed(2) + '°';
      }
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep04(){
    var root = document.getElementById('sim-ep0604');
    if (root) initSim06Ep04(root); else setTimeout(tryInitSim06Ep04, 200);
  }
  tryInitSim06Ep04();
})();
</script>
""")

**Figure 6.4:** Simulateur EP06_04 : Estimateur de Pente par Médiane Angulaire


<figure id="fig-06-sim-ep0604">
  <img src="imagens/fig-06-sim-ep0604.png" alt=" Simulateur EP06_04 : Estimateur de Pente par Médiane Angulaire " style="max-width:80%" />
  <figcaption><strong>Figure 6.4:</strong>  Simulateur EP06_04 : Estimateur de Pente par Médiane Angulaire </figcaption>
</figure>

In [ ]:
%%writefile EP06_04.py
# Code Python

In [ ]:
TestSuite("EP06_04.py").run()

### EP06_05 🟠 Normalisation du fond par division (correction de l'éclairage)

Un formulaire a été photographié sous un éclairage non uniforme, ce qui fait qu'un côté de la feuille apparaît plus clair que l'autre. Dans ces conditions, le seuillage global par Otsu peut produire des résultats insatisfaisants, car un seul seuil ne sépare pas correctement le texte et le fond sur toute l'image. La solution présentée dans le chapitre consiste à **normaliser le fond**, en divisant l'image originale par une version fortement lissée d'elle-même, qui représente l'éclairage basse fréquence.

Dans cet exercice, l'image originale et le fond lissé (équivalent au résultat d'un `cv2.GaussianBlur` avec un $\sigma$ élevé) sont déjà fournis. Votre tâche consiste à implémenter l'étape de normalisation qui produit l'image corrigée.

#### 📋 Directives d'implémentation

1. **Dimensions :** Lire les entiers $L$ (lignes) et $C$ (colonnes).
2. **Image originale :** Lire les $L \times C$ valeurs entières de la matrice `img` (intensités entre 0 et 255).
3. **Fond estimé :** Lire les $L \times C$ valeurs entières de la matrice `bg` (intensités entre 0 et 255, toujours strictement supérieures à zéro).
4. **Normalisation :** Pour chaque position $(i,j)$, calculer
$$
\text{valeur}(i,j)=
\frac{\text{img}(i,j)}{\text{bg}(i,j)}\times255.
$$
5. **Arrondi :** Arrondir le résultat à l'entier le plus proche (*round half away from zero*, avec `np.floor(img + 0.5)`).
6. **Saturation :** Limiter la valeur obtenue à l'intervalle $[0,255]$.
7. **Sortie :** Imprimer la matrice `img_norm` résultante.

#### 📌 Contraintes de calcul

* **Division par zéro :** l'entrée garantit $\text{bg}(i,j)>0$ à toutes les positions.
* **Ordre des opérations :** d'abord arrondir, puis appliquer la saturation.
* **Traitement indépendant :** chaque pixel doit être normalisé individuellement, sans utiliser les informations des pixels voisins.

#### 🧠 Fondement théorique

| Situation | Effet de la normalisation |
|---|---|
| $\text{img}(i,j)=\text{bg}(i,j)$ | Résultat égal à $255$, correspondant au fond normalisé. |
| $\text{img}(i,j)<\text{bg}(i,j)$ | Résultat inférieur à $255$, préservant les régions plus sombres, comme le texte. |
| $\text{img}(i,j)>\text{bg}(i,j)$ | Résultat supérieur à $255$, ensuite saturé. |
| Fond avec éclairage non uniforme | La division réduit les variations lentes de l'éclairage, rendant l'image plus homogène. |

La division par le fond estimé réduit les effets de l'éclairage non uniforme et préserve le contraste entre le premier plan et le fond, facilitant les étapes ultérieures de segmentation.

#### 📦 Spécification des entrées et sorties (VPL)

**Entrée :**

* Ligne 1 : Entier $L$.
* Ligne 2 : Entier $C$.
* Les $L$ lignes suivantes : éléments de la matrice `img`.
* Les $L$ lignes suivantes : éléments de la matrice `bg`.

**Sortie :**

* Matrice `img_norm`, avec $L$ lignes et $C$ colonnes, contenant des valeurs entières séparées par des espaces.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 2<br>2<br>60 120<br>180 40<br>100 100<br>200 80 | 153 255<br>230 128 | Les valeurs supérieures à $255$ doivent être saturées ; $180/200\times255=229{,}5$ donne $230$ après arrondi. |
| 1<br>3<br>30 60 90<br>60 60 60 | 128 255 255 | Seule la première valeur reste inférieure à $255$ après la normalisation. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0605" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0605 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0605 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0605 button:hover { background: #e8dfcf; }
  #sim-ep0605 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0605_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0605_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim05_ep05_cell { width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 9px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP06_05 : Normalisation de fond par division</span>
  <span class="sim-ep0605_pill">(img / bg) &times; 255</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0605_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Intensité du fond à gauche (bg_esq) : <span id="sim-ep0605_vl" style="font-family:monospace; color:#26241d;">100</span>
      </label>
    </div>
    
    <input id="sim-ep0605_sl" type="range" min="40" max="220" step="5" value="100">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajustez le gradient de fond (gauche &rarr; droite) et observez comment la division annule la variation d'éclairage.
    </div>
  </div>

  <!-- Exibição das Matrizes 1x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        img (Original)
      </div>
      <div id="sim-ep0605_g_img" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        bg (Fond lissé)
      </div>
      <div id="sim-ep0605_g_bg" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        img_norm (Sortie)
      </div>
      <div id="sim-ep0605_g_out" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0605_debug" class="sim-ep0605_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep05(root){
    if (!root || root.dataset.sim06Ep05Init) return;
    root.dataset.sim06Ep05Init = "1";

    var linha_img = [90, 90, 90, 90];
    var slEl = root.querySelector('#sim-ep0605_sl');
    var vlEl = root.querySelector('#sim-ep0605_vl');
    var gImg = root.querySelector('#sim-ep0605_g_img');
    var gBg  = root.querySelector('#sim-ep0605_g_bg');
    var gOut = root.querySelector('#sim-ep0605_g_out');
    var dbg  = root.querySelector('#sim-ep0605_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }

    function cellStyle(v){
      var g = Math.max(0, Math.min(255, v));
      return 'background:rgb(' + g + ',' + g + ',' + g + '); color:' + (g > 140 ? '#000000' : '#ffffff') + ';';
    }

    function render(){
      var bgEsq = parseInt(slEl.value, 10);
      vlEl.textContent = bgEsq;

      // Gradiente linear de bgEsq até 200 na direita, 4 colunas
      var bg = [];
      for (var j = 0; j < 4; j++){
        bg.push(Math.round(bgEsq + (200 - bgEsq) * j / 3));
      }

      gImg.innerHTML = '';
      gBg.innerHTML  = '';
      gOut.innerHTML = '';
      
      var out = [];
      for (var j = 0; j < 4; j++){
        var v = (linha_img[j] / bg[j]) * 255;
        var r = roundHalfAway(v);
        var sat = Math.max(0, Math.min(255, r));
        out.push(sat);

        var ci = document.createElement('div');
        ci.className = 'sim05_ep05_cell';
        ci.style.cssText = cellStyle(linha_img[j]);
        ci.textContent = linha_img[j];
        gImg.appendChild(ci);

        var cb = document.createElement('div');
        cb.className = 'sim05_ep05_cell';
        cb.style.cssText = cellStyle(bg[j]);
        cb.textContent = bg[j];
        gBg.appendChild(cb);

        var co = document.createElement('div');
        co.className = 'sim05_ep05_cell';
        co.style.cssText = cellStyle(sat);
        co.textContent = sat;
        gOut.appendChild(co);
      }

      dbg.textContent = 'bg = [' + bg.join(', ') + ']  |  img_norm = [' + out.join(', ') + ']';
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep05(){
    var root = document.getElementById('sim-ep0605');
    if (root) initSim06Ep05(root); else setTimeout(tryInitSim06Ep05, 200);
  }
  tryInitSim06Ep05();
})();
</script>
""")

**Figure 6.5:** Simulateur EP06_05: Normalisation du fond par division


<figure id="fig-06-sim-ep0605">
  <img src="imagens/fig-06-sim-ep0605.png" alt=" Simulateur EP06_05: Normalisation du fond par division " style="max-width:80%" />
  <figcaption><strong>Figure 6.5:</strong>  Simulateur EP06_05: Normalisation du fond par division </figcaption>
</figure>

In [ ]:
%%writefile EP06_05.py
# Code Python

In [ ]:
TestSuite("EP06_05.py").run()

### EP06_06 🔴 Carte de Variance Locale pour la Détection de Texture

Une usine de tissus doit inspecter des rouleaux de tissu en temps réel, sans disposer d'une image de référence — chaque rouleau présente de petites variations naturelles. Dans cette situation, la stratégie présentée dans le chapitre consiste à analyser l'**homogénéité locale de la texture** : les régions uniformes présentent une faible variance d'intensité dans de petits voisinages, tandis que les rayures, taches et défauts de fabrication produisent des augmentations locales de cette variance.

Dans cet exercice, vous implémenterez le cœur de cette méthode, en calculant la variance locale dans une fenêtre glissante et en générant un masque binaire qui identifie les régions dont la variance dépasse un seuil.

#### 📋 Directives d'Implémentation

1. **Dimensions et paramètres :** Lire les entiers $L$, $C$, $k$ (taille de la fenêtre, toujours impaire) et $T$ (seuil de variance).
2. **Image :** Lire les $L \times C$ valeurs entières de la matrice de texture (intensités entre 0 et 255).
3. **Traitement des bords :** Lorsque la fenêtre dépasse les limites de l'image, utiliser la **réplication de bord**, c'est-à-dire répéter la valeur du pixel valide le plus proche.
4. **Moyenne locale :** Pour chaque position $(i,j)$, calculer
$$
\mu(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{fenêtre}}
\text{texture}(p,q).
$$
5. **Variance locale :** Calculer la variance populationnelle de la fenêtre,
$$
\sigma^2(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{fenêtre}}
\left(\text{texture}(p,q)-\mu(i,j)\right)^2,
$$
ou, de manière équivalente,
$$
\sigma^2(i,j)=\overline{x^2}-\mu(i,j)^2,
$$
où $\overline{x^2}$ représente la moyenne des carrés des intensités.

6. **Arrondi :** Arrondir la variance à l'entier le plus proche (*round half away from zero*, avec `np.floor(res_norm + 0.5)`).

7. **Seuillage :** Définir $\text{masque}(i,j)=1$ si la variance arrondie est **strictement supérieure** à $T$ ; sinon, définir $\text{masque}(i,j)=0$.

8. **Sortie :** Imprimer le masque binaire résultant.

#### 📌 Contraintes Computationnelles

* **Réplication de bord :** utiliser la valeur du pixel valide le plus proche chaque fois que la fenêtre dépasse les limites de l'image.
* **Variance populationnelle :** utiliser le dénominateur $k^2$, jamais $k^2-1$.
* **Comparaison stricte :** le masque doit être calculé en utilisant la condition $\sigma^2_{\text{arrondi}}>T$.
* **Fenêtre impaire :** la valeur de $k$ est toujours impaire, garantissant un pixel central.

#### 🧠 Fondement Théorique

| Situation | Variance locale | Interprétation |
|---|---|---|
| Région uniforme | Faible | Intensités similaires dans le voisinage. |
| Région contenant un défaut | Élevée | La présence d'intensités distinctes augmente la dispersion des valeurs. |
| Petite fenêtre | Plus grande sensibilité aux détails et au bruit | Détecte les altérations localisées. |
| Grande fenêtre | Réponse plus lisse | Met en évidence les défauts plus grands, mais réduit la précision de leur localisation. |

La variance locale mesure la dispersion des intensités dans un voisinage. Les régions homogènes présentent une faible variance, tandis que les altérations de la texture augmentent cette mesure, permettant d'identifier d'éventuels défauts par un simple seuillage.

#### 📦 Spécification d'Entrée et de Sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $L$.
* Ligne 2 : Entier $C$.
* Ligne 3 : Entier $k$ (impair).
* Ligne 4 : Entier $T$.
* Lignes suivantes $L$ : éléments entiers de la matrice de texture.

**Sortie :**

* Masque binaire (valeurs 0 ou 1), avec $L$ lignes et $C$ colonnes.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 3<br>3<br>3<br>50<br>10 10 10<br>10 10 10<br>10 90 10 | 0 0 0<br>1 1 1<br>1 1 1 | Le défaut augmente la variance dans toutes les fenêtres qui le contiennent. |
| 2<br>2<br>3<br>5<br>100 100<br>100 100 | 0 0<br>0 0 | La texture est uniforme ; la variance est nulle sur toute l'image. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0606" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0606 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0606 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0606 button:hover { background: #e8dfcf; }
  #sim-ep0606 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0606_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0606_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0606_cell { width: 44px; height: 44px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP06_06 : Variance locale (détection de texture)</span>
  <span class="sim-ep0606_pill">&sigma;&sup2; = m&eacute;dia(x&sup2;) &minus; m&eacute;dia(x)&sup2;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0606_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Intensité du défaut (position centrale) : <span id="sim-ep0606_vl_def" style="font-family:monospace; color:#26241d;">90</span>
      </label>
    </div>
    <input id="sim-ep0606_sl_def" type="range" min="10" max="255" step="5" value="90">

    <div style="display:flex; justify-content:space-between; align-items:center; margin:10px 0 4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Seuil (T) : <span id="sim-ep0606_vl_t" style="font-family:monospace; color:#26241d;">50</span>
      </label>
    </div>
    <input id="sim-ep0606_sl_t" type="range" min="0" max="2000" step="10" value="50">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajustez la valeur du défaut et le seuil T ; observez comment la fenêtre 3&times;3 propage la détection dans le voisinage.
    </div>
  </div>

  <!-- Exibição das Grades 3x3 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0606_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Texture (3&times;3)
      </div>
      <div id="sim-ep0606_g_tex" style="display:grid; grid-template-columns:repeat(3, 44px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0606_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Masque de défaut
      </div>
      <div id="sim-ep0606_g_mask" style="display:grid; grid-template-columns:repeat(3, 44px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0606_debug" class="sim-ep0606_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep06(root){
    if (!root || root.dataset.sim06Ep06Init) return;
    root.dataset.sim06Ep06Init = "1";

    var slDef = root.querySelector('#sim-ep0606_sl_def');
    var vlDef = root.querySelector('#sim-ep0606_vl_def');
    var slT   = root.querySelector('#sim-ep0606_sl_t');
    var vlT   = root.querySelector('#sim-ep0606_vl_t');
    var gTex  = root.querySelector('#sim-ep0606_g_tex');
    var gMask = root.querySelector('#sim-ep0606_g_mask');
    var dbg   = root.querySelector('#sim-ep0606_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }

    function clampIdx(v, n){
      return Math.max(0, Math.min(n - 1, v));
    }

    function render(){
      var defeito = parseInt(slDef.value, 10);
      var T       = parseInt(slT.value, 10);
      vlDef.textContent = defeito;
      vlT.textContent   = T;

      var N = 3;
      var tex = [[10, 10, 10], [10, defeito, 10], [10, 10, 10]];

      gTex.innerHTML  = '';
      gMask.innerHTML = '';
      
      var mask = [];
      for (var i = 0; i < N; i++){
        var row = [];
        for (var j = 0; j < N; j++){
          var vals = [];
          for (var di = -1; di <= 1; di++){
            for (var dj = -1; dj <= 1; dj++){
              var pi = clampIdx(i + di, N);
              var pj = clampIdx(j + dj, N);
              vals.push(tex[pi][pj]);
            }
          }
          var mean = vals.reduce(function(a, b){ return a + b; }, 0) / vals.length;
          var meanSq = vals.reduce(function(a, b){ return a + b * b; }, 0) / vals.length;
          var varr = meanSq - mean * mean;
          var varRound = roundHalfAway(varr);
          row.push(varRound > T ? 1 : 0);
        }
        mask.push(row);
      }

      var total = 0;
      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var g = tex[i][j];
          var ct = document.createElement('div');
          ct.className = 'sim-ep0606_cell';
          ct.style.cssText = 'background:rgb(' + g + ',' + g + ',' + g + '); color:' + (g > 140 ? '#000000' : '#ffffff') + ';';
          ct.textContent = g;
          gTex.appendChild(ct);

          var m = mask[i][j];
          if (m) total++;

          var cm = document.createElement('div');
          cm.className = 'sim-ep0606_cell';
          if (m) {
            cm.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          } else {
            cm.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
          }
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }

      if (total > 0) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'défaut = ' + defeito + '  |  T = ' + T + '  |  Pixels marcados: ' + total + ' / 9';
    }

    slDef.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep06(){
    var root = document.getElementById('sim-ep0606');
    if (root) initSim06Ep06(root); else setTimeout(tryInitSim06Ep06, 200);
  }
  tryInitSim06Ep06();
})();
</script>
""")

**Figure 6.6:** Simulateur EP06_06 : Carte de variance locale pour la détection de texture


<figure id="fig-06-sim-ep0606">
  <img src="imagens/fig-06-sim-ep0606.png" alt=" Simulateur EP06_06 : Carte de variance locale pour la détection de texture " style="max-width:80%" />
  <figcaption><strong>Figure 6.6:</strong>  Simulateur EP06_06 : Carte de variance locale pour la détection de texture </figcaption>
</figure>

In [ ]:
%%writefile EP06_06.py
# Code Python

In [ ]:
TestSuite("EP06_06.py").run()

### EP06_07 🟣 *Pipeline* d'inspection industrielle : enregistrement par translation et soustraction

Sur une chaîne de production, une caméra fixe photographie chaque pièce passant sur le tapis roulant, en la comparant à une image de référence sans défauts. Le problème : de petites vibrations du tapis déplacent la pièce par rapport à la position de référence à chaque capture. Si la soustraction d'images est appliquée directement, sans correction, le déplacement en lui-même génère déjà d'énormes différences — **faux positifs** qui masquent les défauts réels.

Cet exercice est le plus complet du chapitre : vous devez **d'abord enregistrer** (aligner géométriquement) l'image capturée en utilisant un déplacement connu $(dx, dy)$, fourni par un capteur de position du tapis, et **seulement ensuite appliquer la soustraction** avec seuillage, exactement comme décrit dans la section sur l'inspection industrielle.

#### 📋 Directives d'implémentation

1. **Dimensions et paramètres :** Lire $L$, $C$ (dimensions des images), le déplacement entier connu $dx, dy$ (pouvant être négatifs) et le seuil de détection $T$ (entier).
2. **Images :** Lire la matrice de référence (`ref`, $L\times C$, sans défauts) et la matrice capturée (`cap`, $L\times C$, éventuellement déplacée et avec défaut).
3. **Enregistrement par translation :** Construire l'image alignée `alin` en appliquant le déplacement $(dx,dy)$ reçu :
$$
\text{alin}(i,j) = \begin{cases} \text{cap}(i+dy,\; j+dx), & \text{si } (i+dy,\ j+dx) \in [0,L)\times[0,C) \\ 0, & \text{sinon} \end{cases}
$$
4. **Remplissage des bords :** Les positions qui « sortent » de l'image capturée après le déplacement reçoivent la valeur **0** (*zero-padding* — hors du champ de vision de la caméra ; **notez que cet exercice utilise zéro, contrairement à la réplication des bords de l'EP06_06**).
5. **Différence absolue :** Calculer, pixel par pixel,
$$
\text{diff}(i,j) = |\text{ref}(i,j) - \text{alin}(i,j)|
$$
6. **Seuillage :** Définir $\text{masque}(i,j) = 1$ si $\text{diff}(i,j) > T$ ; sinon, $\text{masque}(i,j) = 0$.
7. **Sortie :** Dans cet ordre — (a) la matrice `alin` ($L\times C$) ; (b) le masque de défaut ($L\times C$) ; (c) une dernière ligne avec le nombre total de pixels classés comme défectueux.

#### 📌 Contraintes computationnelles

* ***Zero-padding*, pas de réplication :** les positions hors des limites de l'image capturée, après le déplacement, valent exactement 0 — c'est le point qui différencie le plus cet exercice de l'EP06_06.
* **Comparaison stricte :** $\text{diff}(i,j) > T$.
* **Signe de $(dx,dy)$ :** le déplacement peut être positif ou négatif ; la formule de l'étape 3 doit être appliquée littéralement, sans inverser les signes.
* **Toutes les valeurs sont des entiers :** aucun arrondi n'est effectué à cette étape.

#### 🧠 Fondement théorique

| Étape omise | Conséquence |
|---|---|
| Sauter l'enregistrement géométrique | Le bord entier de l'image (introduit par le déplacement) est marqué comme « défaut » — faux positif systématique |
| Enregistrement avec $(dx,dy)$ incorrect | La pièce et la référence restent désalignées ; la soustraction détecte des contours déplacés, pas de véritables défauts |
| Seuil $T$ trop bas | Le bruit de capture (variations de 1 à 2 niveaux de gris) est confondu avec un défaut |
| Seuil $T$ trop élevé | Les défauts subtils ne sont plus détectés |

L'enregistrement géométrique et la soustraction sont des étapes complémentaires : le premier garantit que les deux images représentent exactement la même scène dans le même référentiel spatial ; la seconde isole ce qui a réellement changé entre elles — idéalement, uniquement les défauts.

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $L$.
* Ligne 2 : Entier $C$.
* Ligne 3 : Deux entiers $dx$ et $dy$, séparés par un espace.
* Ligne 4 : Entier $T$.
* Les $L$ lignes suivantes : éléments entiers de la matrice `ref`.
* Les $L$ lignes suivantes : éléments entiers de la matrice `cap`.

**Sortie :**

* $L$ lignes avec la matrice `alin`.
* $L$ lignes avec le masque de défaut (0/1).
* Dernière ligne : `Total de pixels défectueux : X`.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 3<br>3<br>1 0<br>30<br>50 50 50<br>50 50 50<br>50 50 50<br>0 50 50<br>0 50 90<br>0 50 50 | 50 50 0<br>50 90 0<br>50 50 0<br>0 0 1<br>0 1 1<br>0 0 1<br>Total de pixels défectueux : 4 | $dx=1$ décale la lecture d'une colonne vers la droite ; la dernière colonne de `alin` n'a pas de correspondance <br> (devient 0) et est systématiquement marquée ; le défaut réel (90) est également détecté. |
| 2<br>2<br>0 0<br>20<br>10 10<br>10 10<br>10 10<br>10 60 | 10 10<br>10 60<br>0 0<br>0 1<br>Total de pixels défectueux : 1 | Sans déplacement ($dx=dy=0$) : `alin` est identique à `cap` ; seul le défaut réel (60) est détecté. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0607" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0607 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0607 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0607 button:hover { background: #e8dfcf; }
  #sim-ep0607 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0607_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0607_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0607_cell { width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 10px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP06_07 : Registre par Translation + Soustraction</span>
  <span class="sim-ep0607_pill">|ref &minus; alin(dx,dy)| &gt; T</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0607_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Déplacement horizontal (dx) : <span id="sim-ep0607_vl_dx" style="font-family:monospace; color:#26241d;">1</span>
      </label>
    </div>
    <input id="sim-ep0607_sl_dx" type="range" min="-2" max="2" step="1" value="1">

    <div style="display:flex; justify-content:space-between; align-items:center; margin:10px 0 4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Seuil (T) : <span id="sim-ep0607_vl_t" style="font-family:monospace; color:#26241d;">30</span>
      </label>
    </div>
    <input id="sim-ep0607_sl_t" type="range" min="0" max="100" step="5" value="30">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajustez le déplacement du tapis (dx) et le seuil T. Observez comment le bord « fantôme » disparaît lorsque dx = 0.
    </div>
  </div>

  <!-- Exibição das Grades 3x3 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        ref
      </div>
      <div id="sim-ep0607_g_ref" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        alin (enregistré)
      </div>
      <div id="sim-ep0607_g_alin" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        masque
      </div>
      <div id="sim-ep0607_g_mask" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0607_debug" class="sim-ep0607_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep07(root){
    if (!root || root.dataset.sim06Ep07Init) return;
    root.dataset.sim06Ep07Init = "1";

    var N = 3;
    var ref = [[50, 50, 50], [50, 50, 50], [50, 50, 50]];
    // cap representa a peça deslocada 1 px à direita (col 0 = 0) mais um defeito em (1,2)
    var cap = [[0, 50, 50], [0, 50, 90], [0, 50, 50]];

    var slDx  = root.querySelector('#sim-ep0607_sl_dx');
    var vlDx  = root.querySelector('#sim-ep0607_vl_dx');
    var slT   = root.querySelector('#sim-ep0607_sl_t');
    var vlT   = root.querySelector('#sim-ep0607_vl_t');
    var gRef  = root.querySelector('#sim-ep0607_g_ref');
    var gAlin = root.querySelector('#sim-ep0607_g_alin');
    var gMask = root.querySelector('#sim-ep0607_g_mask');
    var dbg   = root.querySelector('#sim-ep0607_debug');

    function cellStyle(g){
      var v = Math.max(0, Math.min(255, g));
      return 'background:rgb(' + v + ',' + v + ',' + v + '); color:' + (v > 140 ? '#000000' : '#ffffff') + ';';
    }

    function render(){
      var dx = parseInt(slDx.value, 10);
      var T  = parseInt(slT.value, 10);
      vlDx.textContent = dx;
      vlT.textContent  = T;

      gRef.innerHTML  = '';
      gAlin.innerHTML = '';
      gMask.innerHTML = '';

      var alin = [], mask = [], total = 0;

      for (var i = 0; i < N; i++){
        var rowA = [], rowM = [];
        for (var j = 0; j < N; j++){
          var pj = j + dx;
          var v = (pj >= 0 && pj < N) ? cap[i][pj] : 0;
          rowA.push(v);

          var diff = Math.abs(ref[i][j] - v);
          var m = diff > T ? 1 : 0;
          if (m) total++;
          rowM.push(m);
        }
        alin.push(rowA);
        mask.push(rowM);
      }

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var cr = document.createElement('div');
          cr.className = 'sim-ep0607_cell';
          cr.style.cssText = cellStyle(ref[i][j]);
          cr.textContent = ref[i][j];
          gRef.appendChild(cr);

          var ca = document.createElement('div');
          ca.className = 'sim-ep0607_cell';
          ca.style.cssText = cellStyle(alin[i][j]);
          ca.textContent = alin[i][j];
          gAlin.appendChild(ca);

          var m = mask[i][j];
          var cm = document.createElement('div');
          cm.className = 'sim-ep0607_cell';
          if (m) {
            cm.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          } else {
            cm.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
          }
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }

      if (total > 0) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'dx = ' + dx + '  |  T = ' + T + '  |  Total de pixels defeituosos: ' + total + ' / 9';
    }

    slDx.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep07(){
    var root = document.getElementById('sim-ep0607');
    if (root) initSim06Ep07(root); else setTimeout(tryInitSim06Ep07, 200);
  }
  tryInitSim06Ep07();
})();
</script>
""")

**Figure 6.7:** Simulateur EP06_07 : *Pipeline* d


<figure id="fig-06-sim-ep0607">
  <img src="imagens/fig-06-sim-ep0607.png" alt=" Simulateur EP06_07 : *Pipeline* d'inspection — Enregistrement par translation et soustraction " style="max-width:80%" />
  <figcaption><strong>Figure 6.7:</strong>  Simulateur EP06_07 : *Pipeline* d'inspection — Enregistrement par translation et soustraction </figcaption>
</figure>

In [ ]:
%%writefile EP06_07.py
# Code Python

In [ ]:
TestSuite("EP06_07.py").run()

### EP06_08 ⚫ Segmentation et Décodage Réel de *QRCode* avec OpenCV

Dans les exercices précédents, les grandeurs intermédiaires du *pipeline* de traitement d'images — telles que les aires, les périmètres, les variances et les déplacements — étaient fournies directement ou calculées à partir de matrices numériques, sans nécessiter de bibliothèques spécialisées de Vision par Ordinateur. Dans cet exercice de clôture du chapitre, cette restriction est levée de manière intentionnelle : la bibliothèque **OpenCV** (`cv2`) sera utilisée pour localiser et décoder un *QRCode* réel présent dans une scène.

La proposition reproduit un flux simplifié de systèmes employés en inspection visuelle, en automatisation industrielle et en lecture automatique de documents. Pour maintenir l'entrée de données accessible au contexte éducatif, le chargement de l'image sera intégré à la bibliothèque didactique `morph`, via la fonction `mm.readImg`.

La scène est fournie au format **PGM ASCII (P2)** et contient un unique *QRCode* valide, ainsi que divers **objets distracteurs**, tels que des rectangles, des zones de bruit texturé et des blocs isolés. La segmentation basée uniquement sur des propriétés géométriques — comme l'aire et la forme approximativement carrée — est nécessaire pour réduire l'espace de recherche, mais ne suffit pas à identifier le code correct. La confirmation finale sera réalisée exclusivement par la tentative de décodage avec `cv2.QRCodeDetector`, procédure compatible avec les applications réelles de reconnaissance automatique.

#### 📋 Directives d'Implémentation

1. **Lecture des dimensions et des paramètres**

   Lire, dans cet ordre, via l'entrée standard :

   - une ligne contenant le nombre de lignes $L$ ;
   - une ligne contenant le nombre de colonnes $C$ ;
   - une ligne contenant les quatre paramètres de l'algorithme séparés par des espaces :
     - le seuil de binarisation $T$ (entier) ;
     - l'aire minimale $A_{\text{min}}$ (entier) ;
     - la tolérance d'aspect $\text{tol}$ (réel) ;
     - la marge $M$ (entier, en pixels).

2. **Chargement de l'image**

   Utiliser la fonction didactique `f = mm.readImg(L, C)` pour lire les $L \times C$ valeurs de l'image en niveaux de gris, obtenant ainsi un *array* NumPy de type `uint8`.

3. **Binarisation**

   Appliquer une binarisation inversée en utilisant le seuil $T$. Tout pixel de l'image originale avec une intensité strictement supérieure à $T$ doit être converti à 255, tandis que les autres doivent prendre la valeur 0.

4. **Détection des contours**

   Extraire les composants connectés externes en utilisant `cv2.findContours(...)` avec les paramètres :

   * `cv2.RETR_EXTERNAL` ;
   * `cv2.CHAIN_APPROX_SIMPLE`.

5. **Filtrage géométrique**

   Pour chaque contour trouvé :

   * calculer le rectangle englobant `(x, y, w, h)` à l'aide de `cv2.boundingRect` ;
   * ne conserver que les candidats satisfaisant simultanément :

     **Aire minimale**

     $$
     w \times h > A_{\text{min}}
     $$

     **Rapport d'aspect**

     $$
     \left|\frac{w}{h}-1\right| \le \text{tol}
     $$

6. **Tri des candidats**

   Trier les candidats par l'aire du rectangle englobant

   $$
   w \times h
   $$

   en ordre décroissant.

   En cas d'égalité, préserver l'ordre originalement retourné par `cv2.findContours`.

7. **Vérification par décodage**

   Pour chaque candidat, en suivant l'ordre établi :

   * étendre le rectangle de $M$ pixels dans les quatre directions ;
   * limiter les indices pour rester dans l'image ;
   * extraire le recadrage directement depuis l'image originale `f` ;
   * appliquer `cv2.QRCodeDetector().detectAndDecode(...)` sur ce recadrage.

8. **Critère d'arrêt**

   Interrompre immédiatement le traitement lorsque le premier candidat produit une *chaîne* décodée non vide.

9. **Cas non trouvé**

   Si aucun candidat n'est décodé avec succès, imprimer exactement : `QRCODE_NAO_ENCONTRADO`

10. **Sortie (cas trouvé)**

    Imprimer deux lignes.

    Première ligne : `linha coluna altura largura` en utilisant le rectangle englobant **original**, avant l'expansion par la marge $M$.

    Deuxième ligne : `texto_decodificado`

#### 📌 Contraintes Computationnelles

* Utiliser les fonctions d'OpenCV pour réaliser la binarisation, la détection des contours, le calcul du rectangle englobant et le décodage du QRCode.
* Le filtrage géométrique doit obligatoirement avoir lieu avant l'étape de décodage.
* Utiliser exclusivement le seuil fixe $T$ fourni en entrée. Il n'est pas permis d'utiliser des méthodes automatiques de seuillage, comme Otsu ou le seuillage adaptatif.
* Garantir que les recadrages envoyés au décodeur restent dans les limites de l'image.

#### 🧠 Fondement Théorique

| Étape                    | Rôle dans le pipeline                                                                                      | Conséquence si omise                                                                    |
| ------------------------ | ---------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------- |
| **Filtrage géométrique** | Réduit l'espace de recherche en sélectionnant uniquement les régions compatibles avec la géométrie attendue d'un QRCode. | Le décodeur traiterait tous les contours, y compris les bruits et les objets distracteurs. |
| **Décodage**             | Confirme sémantiquement si le candidat contient un QRCode valide.                                          | Des objets géométriquement similaires pourraient être classifiés incorrectement comme QRCode. |
| **Marge $M$**            | Préserve la *zone de silence* autour du code, facilitant sa détection.                                     | L'absence de cette marge peut empêcher l'alignement et la lecture correcte du code.        |

Cet exercice intègre les concepts étudiés tout au long du chapitre dans un *pipeline* unique de Vision par Ordinateur. La segmentation réduit l'ensemble des régions candidates grâce à des caractéristiques géométriques, tandis que l'étape de décodage valide le contenu de la région en utilisant un algorithme spécialisé de reconnaissance.

#### 📦 Spécification d'Entrée et de Sortie (VPL)

**Structure d'Entrée**

```
L
C
T A_min tol M
[matrice de l'image]
```

**Structure de Sortie (Succès)**

```
linha coluna altura largura
texto_decodificado
```

**Structure de Sortie (Échec)**

```
QRCODE_NAO_ENCONTRADO
```

#### 📌 Fichiers de Référence (.pgm)

À des fins de validation, de débogage local et d'analyse de matrices réelles de pixels, les fichiers d'image générés au format ASCII P2 sont disponibles dans le répertoire du projet. Vous pouvez les utiliser pour tester la conformité de votre code en les décodant avec votre téléphone (sauvegardez localement les *.pgm pour les visualiser) :

* 📥 **[Cas 1 : Schéma Normal](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso1_Normal.pgm)** – Contient un unique code parfaitement centré avec des distracteurs géométriques simples en périphérie.
* 📥 **[Cas 2 : Scénario Complexe](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso2_Complexo.pgm)** – Présente une densité de bruit texturé plus élevée et de multiples candidats distracteurs qui testent les limites du filtrage par aspect.
* 📥 **[Cas 3 : Message Étendu](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso3_MensagemSecreta.pgm)** – Contient un QRCode structuré à partir d'une chaîne de caractères plus longue, générant une densité de modules internes plus élevée.
* 📥 **[Cas 4 : Géométrie Compacte](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso4_Excelente.pgm)** – Évalue le comportement du pipeline dans des conditions optimisées de contraste et de positionnement limite.
* 📥 **[Cas 5 : Scénario d'Exclusion](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso5_Nao_Encontrado.pgm)** – Image composée purement d'éléments distracteurs de grande aire, conçue pour valider le comportement de défaillance contrôlée du programme.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0608" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<!-- Cabeçalho no padrão institucional -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">📋 Simulateur EP06_08 : Segmentation et Décodage de QRCode</span>
  <span style="font-size:10px;font-weight:700;padding:3px 10px;border-radius:40px;border:1px solid #e4dcc8;background:#26241d;color:#7ee7c6;font-family:monospace;">Filtre géométrique&rarr;Critère d'arrêt sémantique</span>
</div>

<div style="padding:16px;background:#ffffff;">
  <p style="margin:0 0 14px 0;font-size:11px;color:#8a8371;line-height:1.5;text-align:center;font-weight:600;">
    Ajustez interactivement les paramètres d'entrée de l'algorithme (A_min et tol) pour vérifier quels composants sont filtrés géométriquement et comment le critère d'arrêt par analyse sémantique interrompt le balayage de la file.
  </p>
  
  <div style="display:flex;gap:14px;margin-bottom:14px;flex-wrap:wrap;">
    <!-- Slider Area Minima -->
    <div style="flex:1;min-width:200px;background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#5e5a4a;">Aire minimale (A_min, px&sup2;)</label>
        <span id="sim-ep0608_vl_amin" style="font-family:monospace;font-weight:700;color:#26241d;">250</span>
      </div>
      <input id="sim-ep0608_sl_amin" style="width:100%;accent-color:#26241d;cursor:pointer;height:4px;" max="3000" min="0" step="50" type="range" value="250">
    </div>
    
    <!-- Slider Tolerancia -->
    <div style="flex:1;min-width:200px;background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#5e5a4a;">Tolérance d'aspect (tol)</label>
        <span id="sim-ep0608_vl_tol" style="font-family:monospace;font-weight:700;color:#26241d;">0.22</span>
      </div>
      <input id="sim-ep0608_sl_tol" style="width:100%;accent-color:#26241d;cursor:pointer;height:4px;" max="1.0" min="0.05" step="0.01" type="range" value="0.22">
    </div>
  </div>

  <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(240px, 1fr));gap:14px;margin-bottom:14px;">
    <!-- Canvas da Cena -->
    <div style="text-align:center;background:#fafaf7;border:1px solid #e9e3d3;padding:14px;border-radius:12px;">
      <div style="font-size:10px;font-weight:700;color:#8a8371;text-transform:uppercase;margin-bottom:10px;letter-spacing:0.04em;">Visualisation de la Scène (Matrice f)</div>
      <canvas id="sim-ep0608_canvas" width="260" height="260" style="border:1px solid #e4dcc8;border-radius:10px;background:#ffffff;margin:0 auto;display:block;"></canvas>
    </div>
    
    <!-- Lista de Candidatos -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;padding:14px;border-radius:12px;">
      <div style="font-size:10px;font-weight:700;color:#8a8371;text-transform:uppercase;margin-bottom:10px;text-align:center;letter-spacing:0.04em;">Composants Connectés dans la File</div>
      <div id="sim-ep0608_lista" style="font-family:monospace;font-size:11px;display:flex;flex-direction:column;gap:8px;"></div>
    </div>
  </div>
  
  <!-- Console de Saída VPL -->
  <div id="sim-ep0608_debug" style="background:#fafaf7;border-radius:12px;padding:12px;border:1px solid #e9e3d3;font-family:monospace;font-size:11px;color:#26241d;text-align:center;"></div>
</div>

<script>
(function(){
  function initSim06Ep08(root){
    if(!root || root.dataset.sim06Ep08Init) return;
    root.dataset.sim06Ep08Init = "1";

    var formas = [
      {x: 145, y: 35,  w: 76, h: 76, tipo: "Componente QRCode Real", cor: "#cbd5e1", decodifica: true, padrao: "qr"},
      {x: 35,  y: 145, w: 55, h: 68, tipo: "Falso QRCode (Assimétrico)", cor: "#e2e8f0", decodifica: false, padrao: "falso_qr"},
      {x: 45,  y: 35,  w: 44, h: 44, tipo: "Círculo / Distrator", cor: "#f1f5f9", decodifica: false, padrao: "circulo"},
      {x: 160, y: 175, w: 68, h: 26, tipo: "Retângulo Distrator", cor: "#e2e8f0", decodifica: false, padrao: "retangulo"},
      {x: 65,  y: 220, w: 14, h: 14, tipo: "Ruído Isolado", cor: "#f8fafc", decodifica: false, padrao: "ruido"}
    ];

    var slA = root.querySelector('#sim-ep0608_sl_amin');
    var vlA = root.querySelector('#sim-ep0608_vl_amin');
    var slT = root.querySelector('#sim-ep0608_sl_tol');
    var vlT = root.querySelector('#sim-ep0608_vl_tol');
    var canvas = root.querySelector('#sim-ep0608_canvas');
    var ctx = canvas.getContext('2d');
    var lista = root.querySelector('#sim-ep0608_lista');
    var dbg = root.querySelector('#sim-ep0608_debug');

    function desenhaForma(f, estado){
      ctx.save();
      
      var corBorda = '#94a3b8';
      if (estado === 'candidato_ok') corBorda = '#10b981';
      if (estado === 'candidato_falhou') corBorda = '#f43f5e';
      if (estado === 'rejeitado') corBorda = '#cbd5e1';

      ctx.lineWidth = (estado === 'candidato_ok' || estado === 'candidato_falhou') ? 3 : 1.5;
      ctx.strokeStyle = corBorda;

      if (estado === 'rejeitado') {
        ctx.fillStyle = '#f8fafc';
      } else {
        if(f.padrao === 'qr') ctx.fillStyle = '#e2e8f0';
        else if(f.padrao === 'retangulo') ctx.fillStyle = '#fffbeb';
        else if(f.padrao === 'falso_qr') ctx.fillStyle = '#f0f9ff';
        else ctx.fillStyle = '#fdf4ff';
      }

      if(f.padrao === 'circulo'){
        ctx.beginPath();
        ctx.arc(f.x + f.w/2, f.y + f.h/2, f.w/2, 0, 2 * Math.PI);
        ctx.fill(); ctx.stroke();
      } else {
        ctx.fillRect(f.x, f.y, f.w, f.h);
        ctx.strokeRect(f.x, f.y, f.w, f.h);
        
        if(f.padrao === 'qr' || f.padrao === 'falso_qr'){
          var c = f.w / 5;
          ctx.fillStyle = '#ffffff';
          [[f.x + 3, f.y + 3], [f.x + f.w - c - 3, f.y + 3], [f.x + 3, f.y + f.h - c - 3]].forEach(function(p){
            ctx.fillRect(p[0], p[1], c, c);
            ctx.strokeRect(p[0], p[1], c, c);
          });
          
          ctx.fillStyle = (f.padrao === 'qr') ? '#334155' : '#64748b';
          [[f.x + 5, f.y + 5], [f.x + f.w - c + 1, f.y + 5], [f.x + 5, f.y + f.h - c + 1]].forEach(function(p){
            ctx.fillRect(p[0], p[1], c - 4, c - 4);
          });
        }
      }
      ctx.restore();
    }

    function render(){
      var amin = parseInt(slA.value, 10);
      var tol = parseFloat(slT.value);
      vlA.textContent = amin;
      vlT.textContent = tol.toFixed(2);

      ctx.clearRect(0, 0, canvas.width, canvas.height);
      ctx.fillStyle = '#ffffff';
      ctx.fillRect(0, 0, canvas.width, canvas.height);

      var candidatos = formas.map(function(f){
        var area = f.w * f.h;
        var aspecto = f.w / f.h;
        var passaArea = area > amin;
        var passaAspecto = Math.abs(aspecto - 1.0) <= tol;
        return {f: f, area: area, aspecto: aspecto, passa: passaArea && passaAspecto};
      }).sort(function(a, b){ return b.area - a.area; });

      lista.innerHTML = '';
      var encontrado = null;
      var flagParada = false;

      candidatos.forEach(function(c){
        var estado, texto, bgBox, txBox;
        
        if(!c.passa){
          estado = 'rejeitado';
          texto = 'REJEITADO (Área = ' + c.area + ' px&sup2;, Aspeto = ' + c.aspecto.toFixed(2) + ')';
          bgBox = '#f1f5f9';
          txBox = '#94a3b8';
        } else if(flagParada){
          estado = 'rejeitado';
          texto = 'FILA INTERROMPIDA (Critério de Parada Ativo)';
          bgBox = '#f8fafc';
          txBox = '#cbd5e1';
        } else if(c.f.decodifica){
          estado = 'candidato_ok';
          texto = 'SUCESSO: DECODIFICADO &#10004;';
          bgBox = '#ecfdf5';
          txBox = '#059669';
          encontrado = c.f;
          flagParada = true;
        } else {
          estado = 'candidato_falhou';
          texto = 'GEOMETRIA OK &rarr; FALHA NA DECODIFICAÇÃO &#10008;';
          bgBox = '#fff5f5';
          txBox = '#e11d48';
        }
        
        desenhaForma(c.f, estado);
        
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 10px;border-radius:8px;background:' + bgBox + ';border:1px solid #edf2f7;color:' + txBox + ';display:flex;flex-direction:column;gap:2px;';
        
        var nameSpan = document.createElement('strong');
        nameSpan.style.fontSize = '11px';
        nameSpan.textContent = c.f.tipo + ' (' + c.area + ' px²)';
        
        var statusSpan = document.createElement('span');
        statusSpan.style.fontSize = '10px';
        statusSpan.style.opacity = '0.9';
        statusSpan.innerHTML = texto;

        div.appendChild(nameSpan);
        div.appendChild(statusSpan);
        lista.appendChild(div);
      });

      if (encontrado) {
        dbg.style.backgroundColor = '#eafaf1';
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.color = '#04342C';
        dbg.innerHTML = '<div style="text-align:left;font-weight:700;color:#04342C;margin-bottom:4px;">&#128994; SAÍDA PADRÃO (VPL):</div>' +
                        'y=' + encontrado.y + ' x=' + encontrado.x + ' h=' + encontrado.h + ' w=' + encontrado.w + '<br>' +
                        '<span style="color:#04342C;font-weight:700;">"EP06_08 - PDI-VC | Parabens! Voce decodificou este QR Code!"</span>';
      } else {
        dbg.style.backgroundColor = '#fdecea';
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.color = '#c0392b';
        dbg.innerHTML = '<div style="text-align:left;font-weight:700;color:#c0392b;margin-bottom:4px;">&#128308; SAÍDA PADRÃO (VPL):</div>' +
                        'QRCODE_NAO_ENCONTRADO';
      }
    }

    slA.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }
  
  function tryInitSim06Ep08(){
    var root = document.getElementById('sim-ep0608');
    if(root) initSim06Ep08(root); else setTimeout(tryInitSim06Ep08, 200);
  }
  tryInitSim06Ep08();
})();
</script>
</div>
""")

**Figure 6.8:** Simulateur EP06_08 : Segmentation géométrique + Vérification par décodage de QRCode


<figure id="fig-06-sim-ep0608">
  <img src="imagens/fig-06-sim-ep0608.png" alt=" Simulateur EP06_08 : Segmentation géométrique + Vérification par décodage de QRCode " style="max-width:80%" />
  <figcaption><strong>Figure 6.8:</strong>  Simulateur EP06_08 : Segmentation géométrique + Vérification par décodage de QRCode </figcaption>
</figure>

In [ ]:
%%writefile EP06_08.py
# Code Python

In [ ]:
TestSuite("EP06_08.py").run()